In [1]:
import numpy as np
import torch
from datasets import load_dataset


In [2]:
# load the first shard
data_files = {"train": ["data-0000.tar.gz"]}
ds = load_dataset("commaai/commavq", data_files=data_files)


In [3]:
ds

DatasetDict({
    train: Dataset({
        features: ['token.npy', 'pose.npy', 'json', '__key__', '__url__'],
        num_rows: 2500
    })
})

In [4]:
SEGMENT_IDX = 1

In [5]:
poses = np.array(ds["train"][SEGMENT_IDX]["pose.npy"])
tokens = np.array(ds["train"][SEGMENT_IDX]["token.npy"])
tokens = tokens.reshape(tokens.shape[0], -1).astype(np.int64)
tokens = torch.from_numpy(tokens).to(device="cuda")
tokens


tensor([[ 895,  626,  801,  ..., 1019,  548,  558],
        [ 895,  599,  152,  ..., 1019,  968,  378],
        [1019,  203,  576,  ..., 1019,  521,  781],
        ...,
        [ 663,  363,  663,  ..., 1001,   11,  378],
        [ 274,  363,  801,  ..., 1001,  915,  104],
        [ 663,  363,  801,  ...,  597,   11,  156]], device='cuda:0')

In [6]:
tokens[0]

tensor([ 895,  626,  801,  583,   17,  658,  156,  891,  549,  274,  729,  430,
         431,  121,  266,  286,  892,  326,  566,  683,  430,  794,  522,  506,
         981,  663,  860,  546,  213,  300,  708,  154,   73,  905,  729,  976,
         543,  492,  258,  652, 1005,  663,  333,  238,  177,  325,  341,  390,
          59,  952,  887,  964,  923,  118,   97,  315,  772,  727,  677,  402,
         527,  156,  390,  739,  203,  969,  688,  442,  202,  532,  610,  626,
          82,  906,  125,  753,  589,  731,  970,   11,  366,  704,   95,  193,
         137,  667, 1008,  235,  954,   64,  900,  606,   22,  472,   70,   23,
         534,  245,  295,  416,  107,  647,  660,   15,  248,  128,  805,  265,
         866,  849,  567,  471,  464,  230,  483,  926,  814,  813,  306,  177,
         251,  706,  236,  993,  436, 1019,  548,  558], device='cuda:0')

In [7]:
def fn(segment):
    print(segment["json"])
    # raise ValueError("stop")


# ds.map(fn)

## Visualise


In [8]:
import sys

sys.path.append("..")
import numpy as np
import torch
from IPython.display import Video
from tqdm import trange

from utils.video import transpose_and_clip, write_video
from utils.vqvae import CompressorConfig, Decoder

In [9]:
# load model
config = CompressorConfig()
with torch.device("meta"):
    decoder = Decoder(config)
decoder.load_state_dict_from_url(
    "https://huggingface.co/commaai/commavq-gpt2m/resolve/main/decoder_pytorch_model.bin",
    assign=True,
)
decoder = decoder.eval().to(device="cuda")
# decoder

In [10]:
decoded_video = []
with torch.no_grad():
    for i in trange(len(tokens)):
        decoded = decoder(tokens[i][None])
        decoded_video.append(decoded)
decoded_video = torch.cat(decoded_video, dim=0).cpu().numpy()
decoded_video = transpose_and_clip(decoded_video)

100%|██████████| 1200/1200 [00:06<00:00, 182.89it/s]


In [11]:
save_dst = "../examples/data_set_decoded.mp4"
write_video(decoded_video, save_dst, fps=20)

'../examples/data_set_decoded.mp4'

# Arithmetic coding


In [12]:
from collections import Counter

text_sequence = "ABCCD"
counter = Counter(text_sequence)
keys = list(counter.keys())
counts = list(counter.values())
counts = torch.tensor(counts, dtype=torch.float32)
probs = counts / torch.sum(counts)
print("Tokens:", keys)
print("Probabilities:", probs)
lookup = {k: v.item() for k, v in zip(keys, probs)}
lookup

Tokens: ['A', 'B', 'C', 'D']
Probabilities: tensor([0.2000, 0.2000, 0.4000, 0.2000])


{'A': 0.20000000298023224,
 'B': 0.20000000298023224,
 'C': 0.4000000059604645,
 'D': 0.20000000298023224}

In [13]:
prob_intervals = {}
current_low = 0.0
for symbol, prob in lookup.items():
    current_high = current_low + prob
    prob_intervals[symbol] = (current_low, current_high)
    current_low = current_high

prob_intervals

{'A': (0.0, 0.20000000298023224),
 'B': (0.20000000298023224, 0.4000000059604645),
 'C': (0.4000000059604645, 0.800000011920929),
 'D': (0.800000011920929, 1.0000000149011612)}

## Encoding loop


In [14]:
# Arithmetic coding everything into a single float

low = 0.0
high = 1.0
for ch in text_sequence:
    current_range = high - low
    print(f"{current_range=:.6f}")
    symbol_low, symbol_high = prob_intervals[ch]

    high = low + (symbol_high * current_range)
    low = low + (symbol_low * current_range)

    print(f"Encoded '{ch}' -> New Window: [{low:.6f}, {high:.6f})")


current_range=1.000000
Encoded 'A' -> New Window: [0.000000, 0.200000)
current_range=0.200000
Encoded 'B' -> New Window: [0.040000, 0.080000)
current_range=0.040000
Encoded 'C' -> New Window: [0.056000, 0.072000)
current_range=0.016000
Encoded 'C' -> New Window: [0.062400, 0.068800)
current_range=0.006400
Encoded 'D' -> New Window: [0.067520, 0.068800)


# Exampining GPT


In [15]:
import sys

sys.path.append("..")
import numpy as np
import torch
import torch._dynamo.config
import torch._inductor.config
from tqdm import trange

torch._inductor.config.coordinate_descent_tuning = True
torch._inductor.config.triton.unique_kernel_names = True
torch._inductor.config.fx_graph_cache = (
    True  # Experimental feature to reduce compilation times, will be on by default in future
)

from utils.gpt import GPT, GPTConfig
from utils.video import transpose_and_clip, write_video
from utils.vqvae import CompressorConfig, Decoder

In [16]:
# load model
config = GPTConfig()
with torch.device("meta"):
    model = GPT(config)
model.load_state_dict_from_url(
    "https://huggingface.co/commaai/commavq-gpt2m/resolve/main/pytorch_model.bin",
    assign=True,
)
model = model.eval().to(device="cuda", dtype=torch.bfloat16)

In [17]:
model

GPT(
  (transformer): ModuleDict(
    (wte): Embedding(1025, 1024)
    (wpe): Embedding(2580, 1024)
    (h): ModuleList(
      (0-23): 24 x TransformerBlock(
        (attn): Attention(
          (c_attn): Linear(in_features=1024, out_features=3072, bias=True)
          (c_proj): Linear(in_features=1024, out_features=1024, bias=True)
        )
        (mlp): FeedForward(
          (c_fc): Linear(in_features=1024, out_features=4096, bias=True)
          (c_proj): Linear(in_features=4096, out_features=1024, bias=True)
        )
        (ln_1): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
        (ln_2): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
      )
    )
    (ln_f): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=1024, out_features=1025, bias=False)
)

In [18]:
print(tokens.dtype)
tokens.shape


torch.int64


torch.Size([1200, 128])

In [19]:
bos_vector = torch.ones(tokens.shape[0], dtype=torch.int64, device="cuda") * config.bos_token
bos_vector.shape

tokens_to_encode = torch.hstack([bos_vector[:, None], tokens])
tokens_to_encode.shape

torch.Size([1200, 129])

In [20]:
tokens_to_encode = tokens_to_encode.view(-1)
tokens_to_encode.shape

torch.Size([154800])

In [54]:
tokens_to_encode

tensor([1024,  895,  626,  ...,  597,   11,  156], device='cuda:0')

In [56]:
20 * 129

config.block_size

MAX_CONTEXT = config.block_size - 1

for i in range(0, len(tokens_to_encode))[0:5]:
    start = max(0, i - MAX_CONTEXT)
    x = tokens_to_encode[start:i]
    # print(x)
    if i % 129 == 0:
        print(f"Dont encode {x=}")
        # Dont encode that
        continue

    print(f"Encoding {x=}")
    with torch.no_grad():
        logits = model(x[None])
        probs = torch.nn.functional.softmax(logits[0, -1], dim=-1)
    break

probs.argmax()

Dont encode x=tensor([], device='cuda:0', dtype=torch.int64)
Encoding x=tensor([1024], device='cuda:0')


tensor(1024, device='cuda:0')

In [57]:
probs

tensor([0.0006, 0.0005, 0.0019,  ..., 0.0007, 0.0011, 0.0063], device='cuda:0',
       dtype=torch.bfloat16)

In [58]:
pdf_with_zero = torch.cat([torch.zeros(1, device=probs.device), probs])
pdf_with_zero

tensor([0.0000, 0.0006, 0.0005,  ..., 0.0007, 0.0011, 0.0063], device='cuda:0')

In [59]:
cdf = pdf_with_zero.cumsum(dim=-1).clamp(max=1.0)
cdf

tensor([0.0000e+00, 5.8365e-04, 1.0910e-03,  ..., 9.9262e-01, 9.9367e-01,
        1.0000e+00], device='cuda:0')

In [61]:
cdf_formatted = cdf.unsqueeze(0).unsqueeze(0).cpu()
cdf_formatted

tensor([[[0.0000e+00, 5.8365e-04, 1.0910e-03,  ..., 9.9262e-01,
          9.9367e-01, 1.0000e+00]]])